```
# Lab type:  debug
# Course:    NL301 Natural Language Processing with Python
# Lesson:    04 — TF-IDF and Document Similarity
# Task:      Find and fix three bugs in a document retrieval system.
```

## Setup and reference code

In [ ]:
!pip install scikit-learn numpy --quiet
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus = [
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks with many layers",
    "natural language processing involves text analysis",
    "computer vision recognises objects in images",
    "reinforcement learning trains agents through rewards",
    "supervised learning requires labelled training data",
]
query = "neural network deep learning algorithms"

# Reference — correct retrieval
def retrieve_correct(query, corpus, top_n=3):
    tv = TfidfVectorizer()
    corpus_vecs = tv.fit_transform(corpus)
    query_vec   = tv.transform([query])
    scores = cosine_similarity(query_vec, corpus_vecs)[0]
    top_idx = np.argsort(scores)[-top_n:][::-1]
    for i in top_idx:
        print(f"  {scores[i]:.3f}  {corpus[i]}")

retrieve_correct(query, corpus)


---
## Bug 1: CountVectorizer instead of TfidfVectorizer

Cosine similarity on raw counts is biased towards longer documents. TF-IDF normalises for document length and down-weights common terms.

In [ ]:
# BUG: CountVectorizer used instead of TfidfVectorizer
def retrieve_buggy_1(query, corpus, top_n=3):
    cv = CountVectorizer()                   # ← Bug
    corpus_vecs = cv.fit_transform(corpus)
    query_vec   = cv.transform([query])
    scores = cosine_similarity(query_vec, corpus_vecs)[0]
    top_idx = np.argsort(scores)[-top_n:][::-1]
    for i in top_idx:
        print(f"  {scores[i]:.3f}  {corpus[i]}")

retrieve_buggy_1(query, corpus)


**Explain the bug** — how does raw count similarity differ from TF-IDF similarity, and why is TF-IDF preferred for retrieval?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**What the bug does:** `CountVectorizer` weights every word occurrence equally. Common words present in many documents (e.g., "is", "a", "with") accumulate high counts and dominate cosine similarity, biasing results toward longer documents regardless of relevance.

**Correct approach:** Replace `CountVectorizer()` with `TfidfVectorizer()`. TF-IDF down-weights tokens that appear in many documents via IDF, amplifying rare discriminative terms. For the query "neural network deep learning algorithms", TF-IDF correctly surfaces the deep-learning and neural-network documents because those terms are rare in the corpus and receive high IDF weight.

</details>

In [ ]:
# FIX 1: use TfidfVectorizer
def retrieve_fixed_1(query, corpus, top_n=3):
    tv = TfidfVectorizer()
    corpus_vecs = tv.fit_transform(corpus)
    query_vec   = tv.transform([query])
    scores = cosine_similarity(query_vec, corpus_vecs)[0]
    top_idx = np.argsort(scores)[-top_n:][::-1]
    for i in top_idx:
        print(f"  {scores[i]:.3f}  {corpus[i]}")

retrieve_fixed_1(query, corpus)


---
## Bug 2: Wrong sort direction

`np.argsort` returns indices sorted **ascending** (lowest score first). Taking the first 3 returns the *least* similar documents.

In [ ]:
# BUG: argsort slices the wrong end — lowest-scoring docs selected
def retrieve_buggy_2(query, corpus, top_n=3):
    tv = TfidfVectorizer()
    corpus_vecs = tv.fit_transform(corpus)
    query_vec   = tv.transform([query])
    scores = cosine_similarity(query_vec, corpus_vecs)[0]
    top_idx = np.argsort(scores)[:top_n]    # ← Bug: takes smallest
    for i in top_idx:
        print(f"  {scores[i]:.3f}  {corpus[i]}")

print("All scores (for reference):")
tv = TfidfVectorizer()
cv = tv.fit_transform(corpus)
qv = tv.transform([query])
for i, s in enumerate(cosine_similarity(qv, cv)[0]):
    print(f"  {s:.3f}  {corpus[i]}")
print()
print("Buggy top-3:")
retrieve_buggy_2(query, corpus)


**Explain the bug** — what does `np.argsort(scores)[:3]` return vs `np.argsort(scores)[-3:][::-1]`?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**What the bug does:** `np.argsort(scores)` returns indices sorted ascending — lowest cosine score first. `[:top_n]` takes the first 3 indices, which are the three *least similar* documents. The query about "neural network deep learning" ends up retrieving the most irrelevant documents.

**Correct approach:** `np.argsort(scores)[-top_n:][::-1]` — take the last `top_n` indices (highest cosine scores) and reverse to descending order. Alternatively use `np.argsort(-scores)[:top_n]`.

</details>

In [ ]:
# FIX 2: take the last top_n indices and reverse for descending order
def retrieve_fixed_2(query, corpus, top_n=3):
    tv = TfidfVectorizer()
    corpus_vecs = tv.fit_transform(corpus)
    query_vec   = tv.transform([query])
    scores = cosine_similarity(query_vec, corpus_vecs)[0]
    top_idx = np.argsort(scores)[-top_n:][::-1]
    for i in top_idx:
        print(f"  {scores[i]:.3f}  {corpus[i]}")

retrieve_fixed_2(query, corpus)


---
## Bug 3: Vectorizer fit on corpus + query combined

When the query is included in `fit_transform`, query tokens inflate IDF weights. A rare query term appears to be common (it was seen during fitting), reducing its IDF weight and making it harder to distinguish.

In [ ]:
# BUG: fit on corpus + query together
def retrieve_buggy_3(query, corpus, top_n=3):
    all_docs = corpus + [query]              # ← Bug: query included in fit
    tv = TfidfVectorizer()
    all_vecs = tv.fit_transform(all_docs)
    corpus_vecs = all_vecs[:-1]
    query_vec   = all_vecs[-1]
    scores = cosine_similarity(query_vec, corpus_vecs)[0]
    top_idx = np.argsort(scores)[-top_n:][::-1]
    for i in top_idx:
        print(f"  {scores[i]:.3f}  {corpus[i]}")

print("Buggy results:")
retrieve_buggy_3(query, corpus)


**Explain the bug** — how does including the query in `fit` affect IDF weights for query-specific terms?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**What the bug does:** IDF is computed as `log((1 + N) / (1 + df(t)))`. When the query is included in `fit_transform`, each query-specific rare term is seen in at least one document (the query itself), artificially inflating its document frequency and reducing its IDF weight. Query terms that should be highly distinctive get penalised, distorting similarity scores.

**Correct approach:** Fit the vectorizer on the corpus only (`tv.fit_transform(corpus)`), then transform the query separately with the already-fitted vocabulary (`tv.transform([query])`). The query's tokens are looked up in the corpus vocabulary; their IDF weights reflect only how common they are across corpus documents.

</details>

In [ ]:
# FIX 3: fit on corpus only; transform query separately
def retrieve_fixed_3(query, corpus, top_n=3):
    tv = TfidfVectorizer()
    corpus_vecs = tv.fit_transform(corpus)   # fit on corpus only
    query_vec   = tv.transform([query])      # transform query with fitted vocab
    scores = cosine_similarity(query_vec, corpus_vecs)[0]
    top_idx = np.argsort(scores)[-top_n:][::-1]
    for i in top_idx:
        print(f"  {scores[i]:.3f}  {corpus[i]}")

print("Fixed results:")
retrieve_fixed_3(query, corpus)


---
## Summary

1. Bug 1 fix: ___
2. Bug 2 fix: ___
3. Bug 3 fix: ___

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Bug 1:** Replace `CountVectorizer` with `TfidfVectorizer` so common terms are down-weighted by IDF and rare discriminative terms drive retrieval.
2. **Bug 2:** Slice `np.argsort(scores)[-top_n:][::-1]` to get the highest cosine scores in descending order.
3. **Bug 3:** Fit the vectorizer on the corpus only; use `transform` (not `fit_transform`) on the query so IDF weights are uncontaminated by query tokens.

</details>